In [ ]:
import os
import gc
import math
import datetime
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

import cfospy

In [ ]:
def calc_CT_norm(CT_df, th):
    """Z-score normalize per region and compute per-CT means across 4 days."""
    CTs = np.arange(0, 24, 4)  # [0, 4, ..., 20]
    num_samp = 6               # samples per time point

    # Filter by BH.Q threshold
    df_i = CT_df[CT_df["BH.Q"] < th]
    ids = df_i["id"].tolist()

    print(f"Number of regions (BH.Q < {th}): {len(df_i)}")

    # Extract measurement matrix (columns starting with "CT")
    ct_cols = df_i.columns[df_i.columns.str.startswith("CT")]
    df_all = df_i[ct_cols].to_numpy()

    # Per-row z-score normalization
    mean = df_all.mean(axis=1, keepdims=True)
    std = df_all.std(axis=1, keepdims=True)
    df_i = (df_all - mean) / std

    # Aggregate 4 days for each CT by averaging the corresponding 6-sample blocks
    CT_ms = np.zeros((df_i.shape[0], len(CTs)))
    CT_sds = np.zeros_like(CT_ms)

    block = num_samp
    day_stride = block * 6  # 36 columns per day

    for CT_i in range(len(CTs)):
        start = CT_i * block
        end = (CT_i + 1) * block

        CT_li = (
            df_i[:, start:end]
            + df_i[:, start + day_stride:end + day_stride]
            + df_i[:, start + 2 * day_stride:end + 2 * day_stride]
            + df_i[:, start + 3 * day_stride:end + 3 * day_stride]
        ) / 4

        CT_ms[:, CT_i] = CT_li.mean(axis=1)

    return ids, df_i, CT_ms


def calc_score(sample_names, predicted, fig_op=False):
    """Compute timetable score (sum of min absolute differences under 24h wrap)."""
    # True labels from sample names like "CT0_01" → [0, 4, 8, ...]
    ans = [int(s.split("_")[0][2:]) for s in sample_names]
    ans2 = [(a - 24) if a > 20 else a for a in ans]

    # Consider wrap-around (pred±24) and choose min absolute difference
    predicted = np.asarray(predicted, dtype=np.float32)
    ans2 = np.asarray(ans2, dtype=np.float32)

    diff = np.zeros((3, len(sample_names)), dtype=np.float32)
    diff[0] = np.abs(predicted)
    diff[1] = np.abs(predicted + 24)
    diff[2] = np.abs(predicted - 24)

    diff_a = np.abs(diff - ans2)  # shape (3, N)

    min_diff = np.min(diff_a, axis=0)
    score = float(np.sum(min_diff))
    sortin = np.argmin(diff_a, axis=0)

    predicted_t = [diff[s, n] for n, s in enumerate(sortin)]

    print(f"Score (sum of min diffs): {score}")

    if fig_op == True:
        plt.figure(figsize=(10, 6))
        x = np.arange(len(sample_names))
        plt.scatter(x, ans2, s=5)
        plt.scatter(x, predicted_t, s=6)
        plt.plot(ans2, label="True")
        plt.plot(predicted_t, label="Predicted")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
        plt.yticks(np.arange(0, 28, 4))
        plt.ylim(0, )
        ax = plt.gca()
        ax.axes.xaxis.set_ticklabels([])
        ax.axes.yaxis.set_ticklabels([])
        plt.show()

    return score, min_diff


def diff_plot(min_diff, fig_op=False):
    """Plot mean±SEM of phase prediction errors at each CT bin."""
    CT_li = np.arange(0, 24, 4)
    diff_mean = np.zeros(len(CT_li), dtype=np.float32)
    diff_SD = np.zeros(len(CT_li), dtype=np.float32)

    for i, CT in enumerate(CT_li):
        v = min_diff[i:i+6]
        v = np.append(v, min_diff[i+24:i+30])
        if len(min_diff) >= 73:
            v = np.append(v, min_diff[i+48:i+54])
            v = np.append(v, min_diff[i+72:i+78])
        print(v)
        diff_mean[i] = np.mean(v)
        diff_SD[i] = np.std(v) / np.sqrt(len(v))

    print(f"mean {np.mean(diff_mean)}")
    print(f"SD {np.mean(diff_SD)}")

    if fig_op == True:
        plt.errorbar(CT_li, diff_mean, yerr=diff_SD, capsize=5)
        plt.scatter(CT_li, diff_mean, s=15)
        plt.xticks(np.arange(0, 24, 4))
        plt.ylim(0, 2.0)
        plt.show()

    return float(np.mean(diff_mean))


def rad2ph(rad):
    if math.isnan(rad):
        return np.nan
    else:
        return (round((2*np.pi+rad)*180/np.pi*24/360, 1),  round((rad)*180/np.pi*24/360, 1))[rad>=0]

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"

cfos_dir = os.path.join(src, "circadian_1st", "circadian_1st_Reconst")
savedir = os.path.join(dst, "cfos_app")
figdir = os.path.join(savedir, "figure_articles", "timetable")

In [ ]:
# Collect unique sample names

CT_li = np.arange(0, 48, 4)  # circadian time points (CT0–44, every 4 h)
sample_ids = np.arange(1, 7, 1)

reconsts = os.listdir(cfos_dir)
sample_names = []

for CT in CT_li:
    for sample_id in sample_ids:
        sample = f"CT{CT}_{str(sample_id).zfill(2)}"
        for reconst in reconsts:
            if sample in reconst:
                sample_names.append(sample)

print(len(sample_names))

In [ ]:
# Load atlas data
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")

vx = 50
ca = cfospy.analysis.read_atlas_data(rdir, vx)
print(f"Number of all regions: {len(ca.ID_all)}")

In [ ]:
# Read rhythmicity data

cos_dir = os.path.join(src, "cos_results")
res = "cos.cell_count_ratio_1st2nd_small_ai_fpr0.5.csv"

# Read cosinor test results
ct_path = os.path.join(cos_dir, res)
CT_df = pd.read_csv(ct_path)

CT_df

In [ ]:
# Prepare CYCLOPS input data

data = CT_df.loc[:, CT_df.columns.str.startswith("CT")]
CT_C = [int(col.split("_")[0].replace("CT", "")) % 24 for col in data.columns]
batch_D = (
    (("B" + str(1) + "_") * 72).split("_")[:-1]
    + (("B" + str(2) + "_") * 72).split("_")[:-1]
)

# Transpose data for annotation
data = data.T

# Insert circadian time and batch labels
data.insert(0, "CT_C", CT_C)
data.insert(0, "batch_D", batch_D)

# Transpose back to original orientation
data = data.T

# Add acronyms (region names)
acronyms = ["batch_D", "CT_C"] + CT_df["acronym"].tolist()
data.insert(0, "Gene_Symbol", acronyms)

# Reset index to remove original numbering
data = data.reset_index(drop=True)

data

In [ ]:
# Define sample collection times (0–2π scale)
sample_ind = np.arange(1, len(sample_names) + 1)
coll_t = [i * 2 * np.pi / 24 for i in CT_C]

print(list(sample_ind))
print(coll_t)

In [ ]:
# Select seed regions with significant rhythmicity
CT_df_seed = CT_df[CT_df["BH.Q"] < 0.1]

# Extract acronyms and phases (shifted by π)
seed_acronyms = CT_df_seed["acronym"].tolist()
ph_seed = (CT_df_seed["Ph"] + np.pi).tolist()

# Create a formatted acronym string for display
acronym_str = ",".join(f"\"{a}\"" for a in seed_acronyms)

print(acronym_str)
print(ph_seed)

In [ ]:
# Save data for CYCLOPS
data_dir = os.path.join(savedir, "timetable", "CYCLOPS-2.0-main", "data")
os.makedirs(data_dir, exist_ok=True)

data_path = os.path.join(data_dir, "all_data_cr.csv")
data.to_csv(data_path, index=False)

In [ ]:
# Check CYCLOPS results
res_dir = os.path.join(savedir, "timetable", "CYCLOPS-2.0-main", "results")
dirs = os.listdir(res_dir)

file_li = []
diff_mean_li = []

for i, dir in enumerate(dirs):
    print("i", i)
    fits_dir = os.path.join(res_dir, dir, "Fits")
    
    for dir2 in os.listdir(fits_dir):
        if "Fit_Output" in dir2:
            f2 = os.path.join(fits_dir, dir2)
            if not os.path.exists(f2):
                continue

            df_res = pd.read_csv(f2)
            plt.scatter(df_res["ProjectionX"], df_res["ProjectionY"])
            plt.show()

            mc_ph = df_res["Phases_AG"] - np.pi
            predicted = list(map(rad2ph, mc_ph))
            score, min_diff = calc_score(sample_names + sample_names, predicted)
            diff_mean = diff_plot(min_diff)

            file_li.append(f2)
            diff_mean_li.append(diff_mean)

# Identify file with minimum mean difference
min_in = np.argmin(diff_mean_li)
print("minimum diff:", diff_mean_li[min_in])
print("min file:", file_li[min_in])

In [ ]:
# Projection and difference of max index
target_file = os.path.join(res_dir, "2024-08-16T06_38_00_eigen_max_9_seed_max_CV_0_8_seed_min_CV_0_15_seed_mth_Gene_400", "Fits", "Fit_Output_2024-08-16T06_38_00.csv")

df_res = pd.read_csv(target_file)

plt.scatter(df_res["ProjectionX"], df_res["ProjectionY"])
plt.savefig(os.path.join(figdir, "cyclops_projection.SVG"))
plt.show()

mc_ph = df_res["Phases_AG"] - np.pi
predicted = list(map(rad2ph, mc_ph))
score, min_diff = calc_score(sample_names + sample_names, predicted)
diff_mean = diff_plot(min_diff)

In [ ]:
# Projection annotation with color by phase
x = df_res["ProjectionX"]
y = df_res["ProjectionY"]

# Convert IDs (e.g., "CT20_01") to normalized phase values
CT_li = np.array([int(i.split("_")[0][2:]) for i in df_res["ID"].tolist()])
ph_li = (1 - (CT_li % 24) / 24)
ph_li = [(hue + 1/3 - 1) if (hue + 1/3) > 1 else (hue + 1/3) for hue in ph_li]

# Plot points colored by phase
for i in range(len(df_res)):
    plt.scatter(df_res.iloc[i]["ProjectionX"], df_res.iloc[i]["ProjectionY"], color=hsv_to_rgb([ph_li[i], 1, 1]))

plt.savefig(os.path.join(figdir, "cyclops_projection_annot_color.SVG"))
plt.show()

In [ ]:
# Limit seed genes by coefficient of variation (CV)
min_CV = 0.15
max_CV = 0.8
mth_region = 400  # number of top regions to keep

# Sort by mean value across CT columns
ct_cols = CT_df.columns[CT_df.columns.str.startswith("CT")]
mi = CT_df[ct_cols].mean(axis=1)
max_order_ind = np.argsort(mi)[::-1].tolist()
CT_df_o = CT_df.iloc[max_order_ind].iloc[:mth_region]

# Calculate CV and filter by threshold
cv_df = CT_df_o[ct_cols].std(axis=1) / CT_df_o[ct_cols].mean(axis=1)
ind = np.where((cv_df.to_numpy() <= max_CV) & (cv_df.to_numpy() >= min_CV))
ind

In [ ]:
# Normalize data by mean value across CT columns
ct_cols = CT_df_o.columns[CT_df_o.columns.str.startswith("CT")]
df_all = CT_df_o[ct_cols].to_numpy()

mean_vals = np.mean(df_all, axis=1).reshape(-1, 1)
df_i = (df_all - mean_vals) / mean_vals
df_i

In [ ]:
plt.hist(np.std(df_i, axis=1))

In [ ]:
# Convert model phases to predicted CT labels and evaluate alignment
mc_ph = df["Phases_AG"] - np.pi
predicted = list(map(rad2ph, mc_ph))  # radians -> predicted CT labels
print(predicted)

score, min_diff = calc_score(sample_names + sample_names, predicted)
diff_plot(min_diff)  # visualize per-sample differences

# Scatter plot of projections
plt.scatter(df["ProjectionX"], df["ProjectionY"])
plt.xlim(-8, 8)
plt.ylim(-8, 8)
plt.show()

In [ ]:
# Automatic parameter search for CYCLOPS results
res_dir = os.path.join(savedir, "timetable", "CYCLOPS-2.0-main", "results")

# Default parameter values
eigen_max_d = "5"
mincv_d = "0_14_"
maxcv_d = "0_9_"

# Reference date to filter results
reference_date_str = "2024-06-14T10_30_00"
reference_date = datetime.datetime.strptime(reference_date_str, "%Y-%m-%dT%H_%M_%S")

files = os.listdir(res_dir)

eigen_maxs, min_cvs, max_cvs, scores, diff_mean_li = [], [], [], [], []

for i, text in enumerate(files):
    try:
        # Extract date string from folder name
        date_str = "_".join(text.split("_")[0:3])
        date_ex = datetime.datetime.strptime(date_str, "%Y-%m-%dT%H_%M_%S")
    except ValueError:
        traceback.print_exc()
        continue

    if date_ex >= reference_date:
        # Extract eigen_max value
        eigen_max_match = re.search(r"eigen_max_(\d+)", text)
        eigen_max_value = eigen_max_match.group(1) if eigen_max_match else eigen_max_d

        # Extract seed_max_CV value
        seed_max_cv_match = re.search(r"seed_max_CV_([\d_]+)", text)
        seed_max_cv_value = seed_max_cv_match.group(1) if seed_max_cv_match else maxcv_d

        # Extract seed_min_CV value
        seed_min_cv_match = re.search(r"seed_min_CV_([\d_]+)", text)
        seed_min_cv_value = seed_min_cv_match.group(1) if seed_min_cv_match else mincv_d

        # Record parameter values
        print(f"eigen_max_value: {eigen_max_value}")
        print(f"seed_max_cv_value: {seed_max_cv_value}")
        print(f"seed_min_cv_value: {seed_min_cv_value}")

        eigen_maxs.append(int(eigen_max_value))
        max_cvs.append(float(seed_max_cv_value.replace("_", ".").rstrip(".")))
        min_cvs.append(float(seed_min_cv_value.replace("_", ".").rstrip(".")))

        # Read result files under each folder
        fits_dir = os.path.join(res_dir, text, "Fits")
        for f in os.listdir(fits_dir):
            if "Fit_Output" not in f:
                continue
            path = os.path.join(fits_dir, f)

        df = pd.read_csv(path, index_col=0)
        mc_ph = df["Phases_AG"] - np.pi
        predicted = list(map(rad2ph, mc_ph))

        score, min_diff = calc_score(sample_names + sample_names, predicted)
        scores.append(score)

        diff_mean = diff_plot(min_diff)
        diff_mean_li.append(diff_mean)

# Identify minimum score parameters
print("max", np.min(scores))
minin = np.argmin(scores)
min_eigen_max = eigen_maxs[minin]
min_max_cv = max_cvs[minin]
min_min_cv = min_cvs[minin]

print("min_max_cv", min_max_cv)
print("min_min_cv", min_min_cv)
print("min_eigen_max", min_eigen_max)

In [ ]:
# Prepare and scan CYCLOPS result folders (region-wise parameter search)
res_dir = os.path.join(savedir, "timetable", "CYCLOPS-2.0-main", "results")

# Default parameters embedded in folder names
min_region_d = 3
eigen_max_d = "99"
mincv_d = "0_85_"
maxcv_d = "0_1_"

# Global filter ranges
num_up, num_low = 650, 100
eigen_up, eigen_low = 15, 4
mincv_up, mincv_low = 0.45, 0.05
maxcv_up, maxcv_low = 0.9, 0.5

# Target parameters to match exactly
max_cv_i = 0.8
min_cv_i = 0.15
# eigen_max_i = 9  # (kept commented as in original)
num_r_i = 400

# Reference date to filter result folders
reference_date_str = "2024-08-01T15_00_00"
reference_date = datetime.datetime.strptime(reference_date_str, "%Y-%m-%dT%H_%M_%S")

files = os.listdir(res_dir)

eigen_maxs, min_cvs, max_cvs, num_rs = [], [], [], []
scores, diff_mean_li = [], []

for i, text in enumerate(files):
    try:
        # Extract timestamp prefix from folder name (YYYY-mm-ddTHH_MM_SS_*)
        date_str = "_".join(text.split("_")[0:3])
        date_ex = datetime.datetime.strptime(date_str, "%Y-%m-%dT%H_%M_%S")
    except ValueError:
        traceback.print_exc()
        continue

    if date_ex >= reference_date:
        # Parse parameters from folder name
        seed_min_cv_match = re.search(r"seed_min_CV_([\d_]+)", text)
        seed_min_cv_value = seed_min_cv_match.group(1) if seed_min_cv_match else mincv_d

        eigen_max_match = re.search(r"eigen_max_(\d+)", text)
        eigen_max_value = eigen_max_match.group(1) if eigen_max_match else eigen_max_d

        seed_max_cv_match = re.search(r"seed_max_CV_([\d_]+)", text)
        seed_max_cv_value = seed_max_cv_match.group(1) if seed_max_cv_match else maxcv_d

        seed_mth_Gene_match = re.search(r"seed_mth_Gene_([\d_]+)", text)
        seed_mth_Gene_value = seed_mth_Gene_match.group(1) if seed_mth_Gene_match else mincv_d

        # Log parsed values
        print(f"eigen_max_value: {eigen_max_value}")
        print(f"seed_max_cv_value: {seed_max_cv_value}")
        print(f"seed_min_cv_value: {seed_min_cv_value}")
        print(f"seed_mth_Gene_value: {seed_mth_Gene_value}")

        # Range filters
        if round(float(seed_max_cv_value.replace("_", ".").rstrip(".")), 2) > maxcv_up:
            continue
        if round(float(seed_max_cv_value.replace("_", ".").rstrip(".")), 2) < maxcv_low:
            continue
        if round(float(seed_min_cv_value.replace("_", ".").rstrip(".")), 2) > mincv_up:
            continue
        if round(float(seed_min_cv_value.replace("_", ".").rstrip(".")), 2) < mincv_low:
            continue
        if int(eigen_max_value) > eigen_up:
            continue
        if int(eigen_max_value) < eigen_low:
            continue
        if int(seed_mth_Gene_value) > num_up:
            continue
        if int(seed_mth_Gene_value) < num_low:
            continue

        # Exact-match filters
        if max_cv_i != round(float(seed_max_cv_value.replace("_", ".").rstrip(".")), 2):
            continue
        if min_cv_i != round(float(seed_min_cv_value.replace("_", ".").rstrip(".")), 2):
            continue
        if num_r_i != int(seed_mth_Gene_value):
            continue

        # Accumulate parsed numeric parameters
        eigen_maxs.append(int(eigen_max_value))
        max_cvs.append(float(seed_max_cv_value.replace("_", ".").rstrip(".")))
        min_cvs.append(float(seed_min_cv_value.replace("_", ".").rstrip(".")))
        num_rs.append(int(seed_mth_Gene_value))

        # Read Fit_Output CSV under Fits/
        fits_dir = os.path.join(res_dir, text, "Fits")
        path = None
        for f in os.listdir(fits_dir):
            if "Fit_Output" in f:
                path = os.path.join(fits_dir, f)
        if path is None:
            continue

        df = pd.read_csv(path, index_col=0)
        mc_ph = df["Phases_AG"] - np.pi
        predicted = list(map(rad2ph, mc_ph))

        score, min_diff = calc_score(sample_names + sample_names, predicted)
        diff_mean = diff_plot(min_diff)

        diff_mean_li.append(diff_mean)
        scores.append(score)

# Report best (minimum score) parameters
print("max", np.min(scores))
minin = np.argmin(scores)
min_eigen_max = eigen_maxs[minin]
min_max_cv = max_cvs[minin]
min_min_cv = min_cvs[minin]
min_num_r = num_rs[minin]

print("min_max_cv", min_max_cv)
print("min_min_cv", min_min_cv)
print("min_eigen_max", min_eigen_max)
print("min_num_r", min_num_r)

In [ ]:
# Sort and plot mean differences by number of regions
sortin = np.argsort(num_rs)
diff_mean_sort = np.array(diff_mean_li)[sortin]
num_rs_sort = np.array(num_rs)[sortin]

plt.figure(figsize=(10, 4))
plt.plot(num_rs_sort, diff_mean_sort)
plt.scatter(num_rs_sort, diff_mean_sort)

# Save figure (safe path join)
fig_name = f"cyclops_min_diff_eigenmax{eigen_max_i}_mincv{min_cv_i}_maxcv{max_cv_i}.SVG"
plt.savefig(os.path.join(figdir, fig_name))
plt.show()

In [ ]:
# Sort and plot mean differences by number of regions
sortin = np.argsort(num_rs)
diff_mean_sort = np.array(diff_mean_li)[sortin]
num_rs_sort = np.array(num_rs)[sortin]

plt.figure(figsize=(10, 4))
plt.plot(num_rs_sort, diff_mean_sort)
plt.scatter(num_rs_sort, diff_mean_sort)

# Save figure (safe path join)
fig_name = f"cyclops_min_diff_eigenmax{eigen_max_i}_mincv{min_cv_i}_maxcv{max_cv_i}.SVG"
plt.savefig(os.path.join(figdir, fig_name))
plt.show()